In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import csv

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

project_dir = Path("/home/l2e_ayd/cocoa_research")
final_dir = project_dir / "data/processed/final"

cleaned_csv = final_dir / "cleaned_data.csv"
metadata_json = final_dir / "dataset_metadata.json"

# ------------------------------------------------------------
# Class mapping
# ------------------------------------------------------------

class_names = {
    0: "anthracnose",
    1: "cssvd",
    2: "healthy",
}

# ------------------------------------------------------------
# Hash helper
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Build image-level metadata
# ------------------------------------------------------------

rows = []

split_image_counts = Counter()
class_image_counts = Counter()
annotation_counts = Counter()
source_counts = Counter()

total_images = 0
total_annotations = 0

for split in ["train", "valid", "test"]:

    image_dir = final_dir / split / "images"
    label_dir = final_dir / split / "labels"

    for image_path in sorted(image_dir.iterdir()):

        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            print(f"WARNING: Missing label: {image_path}")
            continue

        classes_in_image = []
        object_count = 0

        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                class_id = int(parts[0])
                classes_in_image.append(class_id)

                annotation_counts[class_id] += 1
                object_count += 1
                total_annotations += 1

        if not classes_in_image:
            continue

        unique_classes = sorted(set(classes_in_image))

        if len(unique_classes) == 1:
            class_id = unique_classes[0]
            image_label = class_names[class_id]
        else:
            image_label = "mixed"

        # Determine source from our filename prefix
        if image_path.name.startswith("roboflow_"):
            source_dataset = "Roboflow Cocoa"
            source_archive = "cocoa.v1i.yolov8"
            source_counts["Roboflow Cocoa"] += 1

        elif image_path.name.startswith("kara_"):
            source_dataset = "KaraAgroAI Cocoa Dataset"
            source_archive = "healthy_02/03/04/05"
            source_counts["KaraAgroAI Cocoa Dataset"] += 1

        else:
            source_dataset = "unknown"
            source_archive = "unknown"

        image_hash = sha256_file(image_path)

        image_id = image_path.stem

        rows.append({
            "image_id": image_id,
            "image_path": str(image_path.relative_to(project_dir)),
            "plant": "cocoa",
            "disease": image_label,
            "label": image_label,
            "source_dataset": source_dataset,
            "source_archive": source_archive,
            "image_hash": image_hash,
            "split": split,
            "object_count": object_count,
        })

        total_images += 1
        split_image_counts[split] += 1

        if image_label in class_names.values():
            class_image_counts[image_label] += 1


# ------------------------------------------------------------
# Write cleaned_data.csv
# ------------------------------------------------------------

fieldnames = [
    "image_id",
    "image_path",
    "plant",
    "disease",
    "label",
    "source_dataset",
    "source_archive",
    "image_hash",
    "split",
    "object_count",
]

with open(cleaned_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)


# ------------------------------------------------------------
# Build dataset metadata
# ------------------------------------------------------------

metadata = {
    "dataset_name": "Cocoa Disease Detection Dataset",
    "version": "1.0",
    "description": (
        "A combined cocoa leaf object-detection dataset assembled from "
        "the KaraAgroAI Cocoa Dataset and a public Roboflow Cocoa dataset "
        "for cocoa disease detection research."
    ),

    "task": "Object Detection",

    "classes": {
        "0": "anthracnose",
        "1": "cssvd",
        "2": "healthy",
    },

    "class_descriptions": {
        "anthracnose": "Cocoa leaf affected by anthracnose.",
        "cssvd": "Cocoa leaf affected by Cocoa Swollen Shoot Virus Disease.",
        "healthy": "Healthy cocoa leaf.",
    },

    "dataset_sources": [
        {
            "name": "The KaraAgroAI Cocoa Dataset",
            "doi": "10.7910/DVN/BBGQSP",
            "license": "CC0 1.0",
            "source_type": "Harvard Dataverse",
        },
        {
            "name": "Roboflow Cocoa Dataset",
            "workspace": "sam-k3mhv",
            "project": "cocoa-zgish",
            "version": 1,
            "license": "CC BY 4.0",
        },
    ],

    "statistics": {
        "total_images": total_images,
        "total_annotations": total_annotations,
        "image_level_classes": dict(class_image_counts),
        "annotation_level_classes": {
            class_names[k]: v
            for k, v in sorted(annotation_counts.items())
        },
        "split_image_counts": dict(split_image_counts),
        "source_image_counts": dict(source_counts),
    },

    "split": {
        "strategy": "Image-level stratified split",
        "train": "70%",
        "validation": "20%",
        "test": "10%",
        "random_seed": 42,
    },

    "annotation_format": "YOLO",
    "image_format": "JPG",

    "quality_control": {
        "image_label_pairs_verified": True,
        "invalid_yolo_annotations": 0,
        "corrupt_images": 0,
        "exact_duplicate_images_after_merge": 0,
        "missing_labels_after_final_split": 0,
        "excluded_karaagroai_images": 81,
        "excluded_karaagroai_reason": {
            "missing_image": 79,
            "no_valid_bounding_box": 2,
        },
    },

    "preprocessing": [
        "Pascal VOC XML annotations from KaraAgroAI were converted to YOLO format.",
        "Roboflow YOLO annotations were retained in YOLO format.",
        "Class labels were normalized to anthracnose, cssvd, and healthy.",
        "Exact image duplicates were checked using SHA-256.",
        "Dataset was split at image level to prevent annotation leakage.",
    ],

    "files": {
        "images_metadata": "cleaned_data.csv",
        "dataset_config": "data.yaml",
        "train_images": "train/images",
        "train_labels": "train/labels",
        "validation_images": "valid/images",
        "validation_labels": "valid/labels",
        "test_images": "test/images",
        "test_labels": "test/labels",
    },
}


# ------------------------------------------------------------
# Write metadata JSON
# ------------------------------------------------------------

with open(metadata_json, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=== DATASET DOCUMENTATION CREATED ===")
print(f"Images metadata: {cleaned_csv}")
print(f"Dataset metadata: {metadata_json}")

print()
print("Total images:", total_images)
print("Total annotations:", total_annotations)

print("\nImage-level classes:")
for cls, count in class_image_counts.items():
    print(f"  {cls}: {count}")

print("\nSplits:")
for split, count in split_image_counts.items():
    print(f"  {split}: {count}")

=== DATASET DOCUMENTATION CREATED ===
Images metadata: /home/l2e_ayd/cocoa_research/data/processed/final/cleaned_data.csv
Dataset metadata: /home/l2e_ayd/cocoa_research/data/processed/final/dataset_metadata.json

Total images: 5583
Total annotations: 13289

Image-level classes:
  healthy: 4635
  anthracnose: 682
  cssvd: 265

Splits:
  train: 3908
  valid: 1116
  test: 559


In [2]:
from pathlib import Path

final_dir = Path("/home/l2e_ayd/cocoa_research/data/processed/final")

for file in [
    final_dir / "cleaned_data.csv",
    final_dir / "dataset_metadata.json",
    final_dir / "data.yaml",
]:
    print(f"{file.name}: {'✓ exists' if file.exists() else '✗ missing'}")

cleaned_data.csv: ✓ exists
dataset_metadata.json: ✓ exists
data.yaml: ✓ exists


In [3]:
import pandas as pd

cleaned = pd.read_csv(
    "/home/l2e_ayd/cocoa_research/data/processed/final/cleaned_data.csv"
)

print(cleaned.head())
print()
print("Rows:", len(cleaned))
print("Columns:", list(cleaned.columns))

                                            image_id  \
0  kara_healthy_02_healthy_02_0157044B-91D0-43F3-...   
1  kara_healthy_02_healthy_02_01686F78-244E-400F-...   
2  kara_healthy_02_healthy_02_02334273-8909-43B9-...   
3  kara_healthy_02_healthy_02_02672A0E-5144-4502-...   
4  kara_healthy_02_healthy_02_033641F8-DB93-46DB-...   

                                          image_path  plant  disease    label  \
0  data/processed/final/train/images/kara_healthy...  cocoa  healthy  healthy   
1  data/processed/final/train/images/kara_healthy...  cocoa  healthy  healthy   
2  data/processed/final/train/images/kara_healthy...  cocoa  healthy  healthy   
3  data/processed/final/train/images/kara_healthy...  cocoa  healthy  healthy   
4  data/processed/final/train/images/kara_healthy...  cocoa  healthy  healthy   

             source_dataset       source_archive  \
0  KaraAgroAI Cocoa Dataset  healthy_02/03/04/05   
1  KaraAgroAI Cocoa Dataset  healthy_02/03/04/05   
2  KaraAgroAI Cocoa 